# 04. Понятные baseline-модели

Popularity, Recent History и Frequent History рассчитываются отдельно и сравниваются на одном test cohort.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Корень проекта: <PROJECT_ROOT>


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем pandas, загрузчики и три итоговые метрики  
**Зачем:** baseline не требует ML-библиотек  
**Что получим:** короткий набор импортов

In [3]:
import numpy as np
import pandas as pd
from IPython.display import display

from fashion_recommender.baselines import popular_items
from fashion_recommender.data import load_transactions
from fashion_recommender.evaluation import hit_rate_at_k, map_at_k, mean_recall_at_k
from fashion_recommender.persistence import load_json

### Пути и параметры

**Что делаем:** проверяем JSON временных окон  
**Зачем:** baseline должен использовать test history, а не все данные  
**Что получим:** готовые пути и константы

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

WINDOWS_PATH = PROCESSED_DIR / "temporal_windows.json"
if not WINDOWS_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден файл: {WINDOWS_PATH}\n"
        "Сначала выполните notebook 03_temporal_validation_colab.ipynb."
    )

K = 12
MAX_EVALUATION_USERS = 2_000
RANDOM_STATE = 42

### Загрузка входов

**Что делаем:** читаем транзакции и границы  
**Зачем:** подготовка отделена от baseline-логики  
**Что получим:** `transactions` и `windows`

In [5]:
transactions = load_transactions(TRANSACTIONS_PATH)
windows = load_json(WINDOWS_PATH)
print("Transactions:", transactions.shape)

Transactions: (1048575, 5)


### Test-границы

**Что делаем:** выбираем test cutoff и конец target  
**Зачем:** все три baseline сравниваются на одной неделе  
**Что получим:** две даты

In [6]:
test_cutoff = pd.Timestamp(windows["test"]["cutoff_date"])
test_end = pd.Timestamp(windows["test"]["target_end_date"])
print("Test:", test_cutoff.date(), "—", test_end.date())

Test: 2019-12-25 — 2019-12-31


### History и target

**Что делаем:** разделяем test-окно  
**Зачем:** baseline видит только прошлое  
**Что получим:** две таблицы и leakage assert

In [7]:
history = transactions[transactions["t_dat"] < test_cutoff].copy()
target = transactions[
    transactions["t_dat"].between(test_cutoff, test_end)
].copy()

assert history["t_dat"].max() < test_cutoff
print("History:", history.shape)
print("Target:", target.shape)

History: (1035285, 5)
Target: (13290, 5)


### Ground truth

**Что делаем:** оставляем известных пользователей и уникальные future pairs  
**Зачем:** cold-start не смешивается с персональными baseline  
**Что получим:** полный словарь ответов

In [8]:
known_users = set(history["customer_id"])
target_evaluation = target[target["customer_id"].isin(known_users)]
target_unique = (
    target_evaluation
    .sort_values("t_dat")
    .drop_duplicates(["customer_id", "article_id"])
)
ground_truth = target_unique.groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()
print("Ground-truth users:", len(ground_truth))

Ground-truth users: 7590


### Воспроизводимый cohort

**Что делаем:** случайно выбираем до 2 000 пользователей  
**Зачем:** результат не зависит от порядка строк словаря  
**Что получим:** `evaluation_users` и `ground_truth_sample`

In [9]:
all_users = np.array(sorted(ground_truth))
sample_size = min(MAX_EVALUATION_USERS, len(all_users))
rng = np.random.default_rng(RANDOM_STATE)
evaluation_users = rng.choice(all_users, size=sample_size, replace=False).tolist()
ground_truth_sample = {
    customer_id: ground_truth[customer_id]
    for customer_id in evaluation_users
}
print("Evaluation users:", len(evaluation_users))

Evaluation users: 2000


### Popularity items

**Что делаем:** считаем глобальный Top-12 по history  
**Зачем:** это самый простой неперсональный baseline  
**Что получим:** список популярных article ID

In [10]:
popular_table = popular_items(history, limit=K)
popular_article_ids = popular_table["article_id"].tolist()
print("Popularity Top-12:", popular_article_ids)

Popularity Top-12: ['0706016001', '0706016002', '0372860001', '0464297007', '0706016003', '0759871002', '0562245046', '0673677002', '0673396002', '0568601006', '0562245001', '0156231001']


### Popularity рекомендации

**Что делаем:** копируем один список каждому пользователю  
**Зачем:** модель не использует customer history  
**Что получим:** словарь рекомендаций

In [11]:
popularity_recommendations = {
    customer_id: popular_article_ids.copy()
    for customer_id in evaluation_users
}
print(popularity_recommendations[evaluation_users[0]])

['0706016001', '0706016002', '0372860001', '0464297007', '0706016003', '0759871002', '0562245046', '0673677002', '0673396002', '0568601006', '0562245001', '0156231001']


### Popularity метрики

**Что делаем:** оцениваем первый baseline  
**Зачем:** он задаёт минимальный ориентир качества  
**Что получим:** `popularity_metrics`

In [12]:
popularity_metrics = {
    "model": "Popularity",
    "Recall@12": mean_recall_at_k(ground_truth_sample, popularity_recommendations, K),
    "MAP@12": map_at_k(ground_truth_sample, popularity_recommendations, K),
    "HitRate@12": hit_rate_at_k(ground_truth_sample, popularity_recommendations, K),
    "users_evaluated": len(ground_truth_sample),
    "average_candidates": K,
    "notes": "Top-12 baseline; common cohort",
}
display(pd.Series(popularity_metrics))

model                                     Popularity
Recall@12                                    0.01625
MAP@12                                      0.004119
HitRate@12                                     0.017
users_evaluated                                 2000
average_candidates                                12
notes                 Top-12 baseline; common cohort
dtype: object


### Небольшой fallback

**Что делаем:** дополняем короткий персональный список popularity  
**Зачем:** каждый baseline должен вернуть K уникальных товаров  
**Что получим:** простую вспомогательную функцию

In [13]:
def fill_with_popularity(personal_items, fallback_items, k=12):
    result = []
    for article_id in list(personal_items) + list(fallback_items):
        if article_id not in result:
            result.append(article_id)
        if len(result) == k:
            break
    return result

### Recent History сортировка

**Что делаем:** ставим самые свежие покупки первыми  
**Зачем:** recency является персональным сигналом  
**Что получим:** отсортированный history cohort

In [14]:
cohort_history = history[
    history["customer_id"].isin(evaluation_users)
].copy()
recent_history = cohort_history.sort_values("t_dat", ascending=False)
display(recent_history.head())

            t_dat  ... sales_channel_id
348200 2019-12-24  ...                1
217916 2019-12-24  ...                1
658752 2019-12-24  ...                2
869060 2019-12-24  ...                2
68942  2019-12-24  ...                1

[5 rows x 5 columns]


### Уникальные recent items

**Что делаем:** убираем повтор одной пары и собираем списки  
**Зачем:** один товар не должен занимать несколько позиций  
**Что получим:** `recent_items_by_user`

In [15]:
recent_unique = recent_history.drop_duplicates(
    ["customer_id", "article_id"]
)
recent_items_by_user = (
    recent_unique
    .groupby("customer_id", sort=False)["article_id"]
    .apply(list)
    .to_dict()
)
display(recent_unique.head())

            t_dat  ... sales_channel_id
348200 2019-12-24  ...                1
217916 2019-12-24  ...                1
658752 2019-12-24  ...                2
869060 2019-12-24  ...                2
68942  2019-12-24  ...                1

[5 rows x 5 columns]


### Recent рекомендации

**Что делаем:** берём recent items и дополняем popularity  
**Зачем:** неактивные пользователи тоже получают K товаров  
**Что получим:** Top-12 Recent History

In [16]:
recent_recommendations = {}
for customer_id in evaluation_users:
    personal_items = recent_items_by_user.get(customer_id, [])
    recent_recommendations[customer_id] = fill_with_popularity(
        personal_items, popular_article_ids, K
    )

print(recent_recommendations[evaluation_users[0]])

['0621381009', '0788324009', '0708379004', '0758222007', '0731608001', '0729318004', '0701014002', '0679278004', '0706016001', '0706016002', '0372860001', '0464297007']


### Recent метрики

**Что делаем:** оцениваем второй baseline отдельно  
**Зачем:** его результат не смешан с Popularity  
**Что получим:** `recent_metrics`

In [17]:
recent_metrics = {
    "model": "Recent Personal History",
    "Recall@12": mean_recall_at_k(ground_truth_sample, recent_recommendations, K),
    "MAP@12": map_at_k(ground_truth_sample, recent_recommendations, K),
    "HitRate@12": hit_rate_at_k(ground_truth_sample, recent_recommendations, K),
    "users_evaluated": len(ground_truth_sample),
    "average_candidates": K,
    "notes": "Top-12 baseline; common cohort",
}
display(pd.Series(recent_metrics))

model                        Recent Personal History
Recall@12                                      0.016
MAP@12                                      0.006149
HitRate@12                                    0.0175
users_evaluated                                 2000
average_candidates                                12
notes                 Top-12 baseline; common cohort
dtype: object


### Frequent History counts

**Что делаем:** считаем число покупок каждой user-item пары  
**Зачем:** частые повторные покупки получают больший приоритет  
**Что получим:** таблицу `frequent_history`

In [18]:
frequent_history = (
    cohort_history
    .groupby(["customer_id", "article_id"], as_index=False)
    .size()
    .rename(columns={"size": "purchase_count"})
)
display(frequent_history.head())

                                         customer_id  ... purchase_count
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1

[5 rows x 3 columns]


### Frequent сортировка

**Что делаем:** сортируем по count, затем по article ID  
**Зачем:** tie-break делает результат воспроизводимым  
**Что получим:** ранжированные пары

In [19]:
frequent_ranked = frequent_history.sort_values(
    ["customer_id", "purchase_count", "article_id"],
    ascending=[True, False, True],
)
frequent_items_by_user = (
    frequent_ranked
    .groupby("customer_id", sort=False)["article_id"]
    .apply(list)
    .to_dict()
)
display(frequent_ranked.head())

                                         customer_id  ... purchase_count
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...              1

[5 rows x 3 columns]


### Frequent рекомендации

**Что делаем:** дополняем персональные списки popularity  
**Зачем:** получаем третий самостоятельный baseline  
**Что получим:** Top-12 Frequent History

In [20]:
frequent_recommendations = {}
for customer_id in evaluation_users:
    personal_items = frequent_items_by_user.get(customer_id, [])
    frequent_recommendations[customer_id] = fill_with_popularity(
        personal_items, popular_article_ids, K
    )

print(frequent_recommendations[evaluation_users[0]])

['0621381009', '0679278004', '0701014002', '0708379004', '0729318004', '0731608001', '0758222007', '0788324009', '0706016001', '0706016002', '0372860001', '0464297007']


### Frequent метрики

**Что делаем:** оцениваем третий baseline отдельно  
**Зачем:** теперь три строки готовы к сравнению  
**Что получим:** `frequent_metrics`

In [21]:
frequent_metrics = {
    "model": "Frequent Personal History",
    "Recall@12": mean_recall_at_k(ground_truth_sample, frequent_recommendations, K),
    "MAP@12": map_at_k(ground_truth_sample, frequent_recommendations, K),
    "HitRate@12": hit_rate_at_k(ground_truth_sample, frequent_recommendations, K),
    "users_evaluated": len(ground_truth_sample),
    "average_candidates": K,
    "notes": "Top-12 baseline; common cohort",
}
display(pd.Series(frequent_metrics))

model                      Frequent Personal History
Recall@12                                      0.016
MAP@12                                      0.006141
HitRate@12                                    0.0175
users_evaluated                                 2000
average_candidates                                12
notes                 Top-12 baseline; common cohort
dtype: object


### Итоговая таблица

**Что делаем:** собираем три заранее рассчитанных словаря метрик  
**Зачем:** здесь нет цикла, скрывающего различия baseline  
**Что получим:** `baseline_metrics`

In [22]:
baseline_metrics = pd.DataFrame([
    popularity_metrics,
    recent_metrics,
    frequent_metrics,
])
display(baseline_metrics)

                       model  ...                           notes
0                 Popularity  ...  Top-12 baseline; common cohort
1    Recent Personal History  ...  Top-12 baseline; common cohort
2  Frequent Personal History  ...  Top-12 baseline; common cohort

[3 rows x 7 columns]


### Сохранение baseline

**Что делаем:** записываем компактную CSV  
**Зачем:** notebook 10 загрузит готовые результаты  
**Что получим:** `baseline_metrics.csv`

In [23]:
baseline_path = REPORT_DIR / "baseline_metrics.csv"
baseline_metrics.to_csv(baseline_path, index=False)
print("Сохранено:", baseline_path)

Сохранено: <PROJECT_ROOT>/reports/tables/baseline_metrics.csv
